# Lawful GEE Candidate Scout — FINAL v6 Request Zones + Quote Comparison

This notebook creates a **lawful desk-based paid imagery shortlist** from public Google Earth Engine layers.

v6 adds only three new capabilities on top of v5:

1. Request-zone creation from the best stable candidates.
2. Stronger road/building false-positive warnings.
3. Paid-imagery quote comparison template and ranking.

It does **not** prove treasure, authorize entry, authorize metal detecting, authorize excavation, or replace permits.


In [1]:
# 1) Install dependencies
# In Colab this may take a minute. If already installed, it is quick.
!pip -q install geemap geopandas shapely pyogrio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.9 MB/s eta 0:00:00


In [2]:
# 2) Imports and configuration

import math
import os
import glob
import zipfile
from datetime import datetime, timezone

import ee
import geemap
import pandas as pd

# Your Earth Engine Cloud Project ID
EE_PROJECT_ID = "test-ecd0d"

# TEST AOI. Replace later with your real lawful AOI.
# Format: [west, south, east, north]
AOI_BBOX = [-7.65, 31.45, -7.55, 31.55]

START_DATE = "2026-01-01"
END_DATE = "2026-12-31"

# Grid size in meters. 1000m is safe for a first run.
GRID_SIZE_M = 1000

TOP_N = 25

# v5/v6 quality-upgrade settings.
# These are warning thresholds, not automatic proof that a cell is bad.
# Adjust only after you understand the effect on your AOI.
BUILTUP_WARNING_FRAC = 0.02       # built-up pixels inside cell
BUILTUP_NEAR_WARNING_FRAC = 0.05  # built-up pixels within nearby buffer
CROPLAND_HEAVY_FRAC = 0.65        # cropland-heavy cells often create false positives
WATER_EDGE_WARNING_FRAC = 0.05    # water-edge/ravine-edge pixels inside cell

# v6 request-zone and stronger false-positive filter settings.
REQUEST_ZONE_TOP_CANDIDATES = 12       # top v6 candidates considered for zoning
REQUEST_ZONE_MAX_ZONES = 6             # keep package small for provider quote
REQUEST_ZONE_CLUSTER_DISTANCE_M = 1800 # merge nearby candidate centers into one request zone
REQUEST_ZONE_BUFFER_M = 450            # buffer around zone candidate centers

# v6 stronger building/road-like warnings.
# These are warning flags only, not proof a candidate is bad.
V6_DYNAMIC_WORLD_BUILT_PROB_THRESHOLD = 0.25
V6_STRONG_BUILT_FRAC = 0.01
V6_BUILDING_NEAR_FRAC = 0.03
V6_ROAD_LIKE_EDGE_FRAC = 0.08
V6_MODERN_CORRIDOR_FRAC = 0.025

# Seasonal windows use month numbers.
# These are generic desk-review windows. For a real AOI, adjust to local dry/wet or low-vegetation seasons.
SEASON_MONTH_WINDOWS = {
    "dry_window": [6, 7, 8, 9],
    "cool_wet_window": [11, 12, 1, 2, 3],
}

print("Project:", EE_PROJECT_ID)
print("AOI_BBOX:", AOI_BBOX)
print("Date range:", START_DATE, "to", END_DATE)
print("Grid size m:", GRID_SIZE_M)
print("Top N:", TOP_N)
print("Season windows:", SEASON_MONTH_WINDOWS)
print("v6 request-zone top candidates:", REQUEST_ZONE_TOP_CANDIDATES)


Project: test-ecd0d
AOI_BBOX: [-7.65, 31.45, -7.55, 31.55]
Date range: 2026-01-01 to 2026-12-31
Grid size m: 1000
Top N: 25
Season windows: {'dry_window': [6, 7, 8, 9], 'cool_wet_window': [11, 12, 1, 2, 3]}
v6 request-zone top candidates: 12


In [3]:
# 3) Authenticate and initialize Earth Engine

try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)

print("Earth Engine initialized")

Earth Engine initialized


In [4]:
# 4) Build AOI and fixed grid

AOI = ee.Geometry.Rectangle(AOI_BBOX, proj="EPSG:4326", geodesic=False)

west, south, east, north = AOI_BBOX
lat_center = (south + north) / 2.0
meters_per_deg_lat = 111_320.0
meters_per_deg_lon = 111_320.0 * math.cos(math.radians(lat_center))

cell_deg_lat = GRID_SIZE_M / meters_per_deg_lat
cell_deg_lon = GRID_SIZE_M / meters_per_deg_lon

x_count = math.ceil((east - west) / cell_deg_lon)
y_count = math.ceil((north - south) / cell_deg_lat)

features = []
for row in range(y_count):
    for col in range(x_count):
        x0 = west + col * cell_deg_lon
        x1 = min(west + (col + 1) * cell_deg_lon, east)
        y0 = south + row * cell_deg_lat
        y1 = min(south + (row + 1) * cell_deg_lat, north)
        geom = ee.Geometry.Rectangle([x0, y0, x1, y1], proj="EPSG:4326", geodesic=False)
        center_lon = (x0 + x1) / 2
        center_lat = (y0 + y1) / 2
        cell_id = f"r{row:04d}_c{col:04d}"
        features.append(
            ee.Feature(
                geom,
                {
                    "cell_id": cell_id,
                    "row": row,
                    "col": col,
                    "center_lon": center_lon,
                    "center_lat": center_lat,
                },
            )
        )

grid = ee.FeatureCollection(features)
grid_count = grid.size().getInfo()

print("Grid cells:", grid_count)
print("Grid columns:", x_count)
print("Grid rows:", y_count)

Grid cells: 120
Grid columns: 10
Grid rows: 12


In [5]:
# 5) Load public GEE datasets and build indices

def mask_s2_sr(image):
    # Sentinel-2 SR Harmonized. Mask common cloud/shadow/snow classes using SCL.
    scl = image.select("SCL")
    mask = (
        scl.neq(3)    # cloud shadow
        .And(scl.neq(8))   # cloud medium probability
        .And(scl.neq(9))   # cloud high probability
        .And(scl.neq(10))  # thin cirrus
        .And(scl.neq(11))  # snow/ice
    )
    scaled = image.select(["B2", "B3", "B4", "B8", "B11", "B12"]).multiply(0.0001)
    return scaled.updateMask(mask).copyProperties(image, image.propertyNames())

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_sr)
)

s2_count_img = s2_col.select("B4").count().rename("s2_count")
s2 = s2_col.median().clip(AOI)

ndvi = s2.normalizedDifference(["B8", "B4"]).rename("ndvi")
mndwi = s2.normalizedDifference(["B3", "B11"]).rename("mndwi")
bsi = (
    s2.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {
            "SWIR": s2.select("B11"),
            "RED": s2.select("B4"),
            "NIR": s2.select("B8"),
            "BLUE": s2.select("B2"),
        },
    )
    .rename("bsi")
)

s1_col = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(["VV", "VH"])
)

s1 = s1_col.median().clip(AOI)
vv_minus_vh = s1.select("VV").subtract(s1.select("VH")).rename("vv_minus_vh")

dem = ee.Image("USGS/SRTMGL1_003").clip(AOI)
slope_deg = ee.Terrain.slope(dem).rename("slope_deg")
tpi_m = dem.subtract(dem.focal_mean(radius=300, units="meters")).rename("tpi_m")

print("Sentinel-2 image count:", s2_col.size().getInfo())
print("Sentinel-1 image count:", s1_col.size().getInfo())

Sentinel-2 image count: 62
Sentinel-1 image count: 44


In [6]:
# 6) Build legal/environmental exclusion masks and v5 false-positive warning layers
# Integrated fixes:
# - JRC water occurrence is unmasked to 0 before thresholding.
# - WorldCover is used for both eligibility and warning fractions.

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(AOI)
    .rename("worldcover_class")
)

# Allow readable land-cover classes:
# 10 trees, 20 shrubland, 30 grassland, 40 cropland, 60 bare/sparse vegetation.
# This is only a desk-triage mask; it is not a legal permission layer.
allowed_wc_classes = [10, 20, 30, 40, 60]
land_ok = (
    worldcover
    .remap(allowed_wc_classes, [1] * len(allowed_wc_classes), 0)
    .unmask(0)
    .rename("land_ok")
    .clip(AOI)
)

surface_water = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
    .unmask(0)
    .gt(10)
    .clip(AOI)
    .rename("surface_water")
)

protected_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(AOI)
protected_count = protected_fc.size().getInfo()

protected_mask = (
    ee.Image(0)
    .byte()
    .paint(protected_fc, 1)
    .clip(AOI)
    .rename("protected_mask")
)

eligible_mask = (
    land_ok.eq(1)
    .And(surface_water.Not())
    .And(protected_mask.Not())
    .rename("eligible_mask")
)

# v5 false-positive warning layers.
# These do not remove cells automatically. They add review warnings.
builtup_mask = worldcover.eq(50).unmask(0).rename("builtup_mask")
cropland_mask = worldcover.eq(40).unmask(0).rename("cropland_mask")

# Nearby built-up and water-edge warnings catch settlement/road/construction or ravine/water-edge artifacts.
builtup_near_mask = (
    builtup_mask
    .focal_max(radius=120, units="meters")
    .rename("builtup_near_mask")
)
water_edge_mask = (
    surface_water
    .focal_max(radius=120, units="meters")
    .And(surface_water.Not())
    .rename("water_edge_mask")
)

print("Protected polygons intersecting AOI:", protected_count)
print("v5 warning layers ready: built-up, cropland-heavy, nearby built-up, water-edge")


Protected polygons intersecting AOI: 0
v5 warning layers ready: built-up, cropland-heavy, nearby built-up, water-edge


In [7]:
# 7) Score pixels as paid archive request candidates

def clamp01(image):
    return image.max(0).min(1)

def local_abs_z(image, name, radius_m=90):
    # Correct Earth Engine implementation:
    # ee.Image has no focal_std(); use reduceNeighborhood(stdDev).
    kernel = ee.Kernel.circle(radius=radius_m, units="meters", normalize=False)
    mean = image.focal_mean(radius=radius_m, units="meters")
    std = image.reduceNeighborhood(
        reducer=ee.Reducer.stdDev(),
        kernel=kernel,
    ).rename(image.bandNames())
    return image.subtract(mean).divide(std.add(0.001)).abs().rename(name)

low_vegetation = clamp01(ee.Image(0.55).subtract(ndvi).divide(0.55)).rename("low_vegetation")
bare_soil_proxy = clamp01(bsi.subtract(-0.10).divide(0.40)).rename("bare_soil_proxy")

visibility_score = (
    low_vegetation.multiply(0.55)
    .add(bare_soil_proxy.multiply(0.45))
    .rename("visibility_score")
)

spectral_contrast = clamp01(local_abs_z(bsi, "spectral_contrast", 90).divide(3.0)).rename("spectral_contrast")
sar_contrast = clamp01(local_abs_z(vv_minus_vh, "sar_contrast", 90).divide(3.0)).rename("sar_contrast")

remote_sensing_contrast = (
    spectral_contrast.multiply(0.70)
    .add(sar_contrast.multiply(0.30))
    .rename("remote_sensing_contrast")
)

gentle_slope_score = clamp01(ee.Image(1).subtract(slope_deg.divide(20))).rename("gentle_slope_score")
tpi_contrast = clamp01(local_abs_z(tpi_m, "tpi_contrast", 150).divide(3.0)).rename("tpi_contrast")

terrain_score = (
    gentle_slope_score.multiply(0.70)
    .add(tpi_contrast.multiply(0.30))
    .rename("terrain_score")
)

# Observation confidence: enough public Sentinel-2 observations.
s2_confidence = clamp01(s2_count_img.divide(8)).rename("s2_confidence")

candidate_score = (
    visibility_score.multiply(0.35)
    .add(remote_sensing_contrast.multiply(0.35))
    .add(terrain_score.multiply(0.20))
    .add(s2_confidence.multiply(0.10))
    .updateMask(eligible_mask)
    .rename("candidate_score")
)

score_stack = ee.Image.cat([
    candidate_score,
    visibility_score,
    remote_sensing_contrast,
    terrain_score,
    low_vegetation,
    bare_soil_proxy,
    spectral_contrast,
    sar_contrast,
    ndvi,
    bsi,
    mndwi,
    slope_deg,
    tpi_m,
    vv_minus_vh,
    s2_count_img,
    land_ok,
    surface_water,
    protected_mask,
    eligible_mask,

    # v5 false-positive warning fractions.
    builtup_mask.rename("builtup_frac"),
    builtup_near_mask.rename("builtup_near_frac"),
    cropland_mask.rename("cropland_frac"),
    water_edge_mask.rename("water_edge_frac"),
]).clip(AOI)

print("Scoring image stack ready")


Scoring image stack ready


In [8]:
# 8) Reduce scores to grid cells and rank candidates
# WorldCover is added as majority/mode, not mean.

def add_worldcover_mode(feature):
    mode = worldcover.reduceRegion(
        reducer=ee.Reducer.mode(),
        geometry=feature.geometry(),
        scale=10,
        maxPixels=1e8,
        tileScale=4,
    ).get("worldcover_class")
    return feature.set("worldcover_mode", mode)

reduced = score_stack.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mean(),
    scale=30,
    tileScale=4,
)

ranked = (
    reduced
    .map(add_worldcover_mode)
    .filter(ee.Filter.notNull(["candidate_score"]))
    .filter(ee.Filter.gt("eligible_mask", 0))
    .sort("candidate_score", False)
)

eligible_count = ranked.size().getInfo()
top_candidates = ranked.limit(TOP_N)

print("Grid cells:", grid_count)
print("Eligible ranked cells:", eligible_count)
print("Top candidates requested:", TOP_N)

Grid cells: 120
Eligible ranked cells: 120
Top candidates requested: 25


In [9]:
# 9) Export basic top candidates to CSV and GeoJSON

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

gdf = geemap.ee_to_gdf(top_candidates)
gdf = gdf.sort_values("candidate_score", ascending=False).reset_index(drop=True)

csv_name = f"lawful_gee_candidate_scout_top_{TOP_N}_{timestamp}.csv"
geojson_name = f"lawful_gee_candidate_scout_top_{TOP_N}_{timestamp}.geojson"

csv_df = gdf.drop(columns=["geometry"], errors="ignore")
csv_df.to_csv(csv_name, index=True)
gdf.to_file(geojson_name, driver="GeoJSON")

print("Saved basic CSV:", csv_name)
print("Saved basic GeoJSON:", geojson_name)
print("Rows:", len(gdf))

display_cols = [
    "cell_id",
    "candidate_score",
    "center_lon",
    "center_lat",
    "eligible_mask",
    "s2_count",
    "visibility_score",
    "remote_sensing_contrast",
    "terrain_score",
    "ndvi",
    "bsi",
    "mndwi",
    "slope_deg",
    "worldcover_mode",
    "builtup_frac",
    "builtup_near_frac",
    "cropland_frac",
    "water_edge_frac",
]
gdf[display_cols].head(10)


Saved basic CSV: lawful_gee_candidate_scout_top_25_20260609T220648Z.csv
Saved basic GeoJSON: lawful_gee_candidate_scout_top_25_20260609T220648Z.geojson
Rows: 25


,cell_id,candidate_score,center_lon,center_lat,eligible_mask,s2_count,visibility_score,remote_sensing_contrast,terrain_score,ndvi,bsi,mndwi,slope_deg,worldcover_mode,builtup_frac,builtup_near_frac,cropland_frac,water_edge_frac
0,r0008_c0008,0.577309,-7.560447,31.526356,0.996287,24.005479,0.742486,0.241125,0.663931,-0.077311,0.106086,0.057371,2.809043,10.0,0.003713,0.094043,0.192265,0
1,r0008_c0007,0.480380,-7.570983,31.526356,1.000000,23.436734,0.589686,0.237660,0.454041,0.140401,0.102587,-0.232920,9.509179,30.0,0.000000,0.000000,0.297836,0
2,r0011_c0008,0.474190,-7.560447,31.549407,1.000000,47.035336,0.534207,0.230709,0.547006,0.276411,0.131224,-0.474653,5.942604,30.0,0.000000,0.000000,0.447907,0
3,r0008_c0001,0.454166,-7.634197,31.526356,0.999233,25.311946,0.497064,0.239366,0.482132,0.271042,0.094546,-0.393389,7.985024,30.0,0.000767,0.037597,0.302026,0
4,r0002_c0009,0.449546,-7.552590,31.472458,1.000000,21.050076,0.438275,0.227801,0.583101,0.343042,0.105615,-0.475703,4.824586,40.0,0.000000,0.000000,0.722328,0
5,r0006_c0007,0.448272,-7.570983,31.508390,1.000000,22.523051,0.540019,0.213240,0.423155,0.276448,0.136615,-0.441273,9.712024,30.0,0.000000,0.000000,0.162125,0
6,r0011_c0001,0.444945,-7.634197,31.549407,0.879534,51.685566,0.416009,0.221679,0.653037,0.312501,0.058552,-0.387531,3.104185,40.0,0.120466,0.255454,0.571963,0
7,r0009_c0005,0.443532,-7.592054,31.535340,1.000000,26.094589,0.540908,0.231741,0.365526,0.244681,0.109653,-0.387414,11.620404,30.0,0.000000,0.000000,0.054565,0
8,r0009_c0004,0.438538,-7.602590,31.535340,1.000000,26.461316,0.519178,0.236094,0.370962,0.244993,0.091825,-0.344983,11.026027,30.0,0.000000,0.000000,0.009741,0
9,r0007_c0008,0.436907,-7.560447,31.517373,0.999233,22.596942,0.348208,0.223975,0.683866,0.340668,0.002577,-0.280863,2.290034,10.0,0.000000,0.000000,0.366647,0


In [10]:
# 10) Visual inspection map

Map = geemap.Map()
Map.centerObject(AOI, 13)

# Public triage layers
Map.addLayer(AOI, {"color": "white"}, "AOI boundary")
Map.addLayer(candidate_score, {"min": 0, "max": 0.8, "palette": ["black", "yellow", "red"]}, "candidate_score")
Map.addLayer(grid.style(color="777777", fillColor="00000000", width=1), {}, "grid")
Map.addLayer(top_candidates.style(color="00FFFF", fillColor="00FFFF33", width=2), {}, "top candidate cells")
Map.addLayer(protected_fc.style(color="FF00FF", fillColor="FF00FF33", width=2), {}, "protected areas intersecting AOI")

# v5 warning context layers. These are review warnings, not automatic rejection.
Map.addLayer(builtup_mask.selfMask(), {"palette": ["red"]}, "warning: built-up pixels", False)
Map.addLayer(cropland_mask.selfMask(), {"palette": ["orange"]}, "warning: cropland pixels", False)
Map.addLayer(water_edge_mask.selfMask(), {"palette": ["blue"]}, "warning: water/ravine edge buffer", False)

Map.addLayerControl()
Map


Map(center=[31.499994328548755, -7.599999999999772], controls=(WidgetControl(options=['position', 'transparent…

In [11]:
# 11) v5 all-cell confidence, score gaps, and false-positive warnings

all_gdf = geemap.ee_to_gdf(ranked)
all_gdf = all_gdf.dropna(subset=[
    "visibility_score",
    "remote_sensing_contrast",
    "terrain_score",
    "s2_count",
    "candidate_score",
]).copy()

all_gdf = all_gdf.sort_values("candidate_score", ascending=False).reset_index(drop=True)
all_gdf["score_rank"] = all_gdf.index + 1
all_gdf["score_percentile"] = all_gdf["candidate_score"].rank(pct=True)

median_score_all = all_gdf["candidate_score"].median()
all_gdf["score_gap_from_median"] = all_gdf["candidate_score"] - median_score_all
all_gdf["next_candidate_score"] = all_gdf["candidate_score"].shift(-1)
all_gdf["score_gap_to_next_rank"] = (
    all_gdf["candidate_score"] - all_gdf["next_candidate_score"]
).fillna(0)

all_gdf["s2_confidence_all"] = (all_gdf["s2_count"] / all_gdf["s2_count"].max()).clip(0, 1)

for col in ["visibility_score", "remote_sensing_contrast", "terrain_score"]:
    all_gdf[col + "_rank_pct"] = all_gdf[col].rank(pct=True)

all_gdf["component_agreement"] = all_gdf[
    [
        "visibility_score_rank_pct",
        "remote_sensing_contrast_rank_pct",
        "terrain_score_rank_pct",
    ]
].mean(axis=1)

all_gdf["confidence_score_all"] = (
    0.35 * all_gdf["score_percentile"]
    + 0.25 * all_gdf["s2_confidence_all"]
    + 0.25 * all_gdf["component_agreement"]
    + 0.15 * (all_gdf["eligible_mask"].clip(0, 1))
)

# v5 false-positive warning flags.
all_gdf["builtup_warning"] = (
    (all_gdf["builtup_frac"] >= BUILTUP_WARNING_FRAC)
    | (all_gdf["builtup_near_frac"] >= BUILTUP_NEAR_WARNING_FRAC)
).astype(int)
all_gdf["cropland_heavy_warning"] = (all_gdf["cropland_frac"] >= CROPLAND_HEAVY_FRAC).astype(int)
all_gdf["water_edge_warning"] = (all_gdf["water_edge_frac"] >= WATER_EDGE_WARNING_FRAC).astype(int)

# This is a conservative heuristic, not a road detector.
# It flags cells that may be dominated by modern linear/edge artifacts.
spectral_q75 = all_gdf["spectral_contrast"].quantile(0.75)
terrain_median = all_gdf["terrain_score"].median()
all_gdf["modern_linear_edge_warning"] = (
    (all_gdf["spectral_contrast"] >= spectral_q75)
    & (all_gdf["terrain_score"] <= terrain_median)
    & (
        (all_gdf["builtup_warning"] == 1)
        | (all_gdf["cropland_heavy_warning"] == 1)
        | (all_gdf["water_edge_warning"] == 1)
    )
).astype(int)

warning_cols = [
    "builtup_warning",
    "cropland_heavy_warning",
    "water_edge_warning",
    "modern_linear_edge_warning",
]
all_gdf["false_positive_warning_count"] = all_gdf[warning_cols].sum(axis=1)

# Warning penalty is intentionally modest. Warnings should trigger visual review, not automatic deletion.
all_gdf["false_positive_penalty"] = (all_gdf["false_positive_warning_count"] * 0.06).clip(0, 0.24)
all_gdf["quality_adjusted_score"] = (
    all_gdf["candidate_score"] * (1 - all_gdf["false_positive_penalty"])
)

quality_cols = [
    "cell_id",
    "score_rank",
    "candidate_score",
    "quality_adjusted_score",
    "confidence_score_all",
    "score_percentile",
    "score_gap_from_median",
    "score_gap_to_next_rank",
    "center_lon",
    "center_lat",
    "s2_count",
    "visibility_score",
    "remote_sensing_contrast",
    "terrain_score",
    "builtup_frac",
    "builtup_near_frac",
    "cropland_frac",
    "water_edge_frac",
    "builtup_warning",
    "cropland_heavy_warning",
    "water_edge_warning",
    "modern_linear_edge_warning",
    "false_positive_warning_count",
    "worldcover_mode",
]

all_gdf.drop(columns=["geometry"], errors="ignore").to_csv("quality_diagnostics_all_cells.csv", index=False)
print("Saved: quality_diagnostics_all_cells.csv")
print("All eligible cells:", len(all_gdf))
all_gdf[quality_cols].head(15)


Saved: quality_diagnostics_all_cells.csv
All eligible cells: 120


,cell_id,score_rank,candidate_score,quality_adjusted_score,confidence_score_all,score_percentile,score_gap_from_median,score_gap_to_next_rank,center_lon,center_lat,...,builtup_frac,builtup_near_frac,cropland_frac,water_edge_frac,builtup_warning,cropland_heavy_warning,water_edge_warning,modern_linear_edge_warning,false_positive_warning_count,worldcover_mode
0,r0008_c0008,1,0.577309,0.542670,0.851046,1.000000,0.180038,0.096929,-7.560447,31.526356,...,0.003713,0.094043,0.192265,0,1,0,0,0,1,10.0
1,r0008_c0007,2,0.480380,0.480380,0.795338,0.991667,0.083109,0.006190,-7.570983,31.526356,...,0.000000,0.000000,0.297836,0,0,0,0,0,0,30.0
2,r0011_c0008,3,0.474190,0.474190,0.883793,0.983333,0.076920,0.020024,-7.560447,31.549407,...,0.000000,0.000000,0.447907,0,0,0,0,0,0,30.0
3,r0008_c0001,4,0.454166,0.454166,0.798140,0.975000,0.056895,0.004620,-7.634197,31.526356,...,0.000767,0.037597,0.302026,0,0,0,0,0,0,30.0
4,r0002_c0009,5,0.449546,0.422573,0.741423,0.966667,0.052275,0.001274,-7.552590,31.472458,...,0.000000,0.000000,0.722328,0,0,1,0,0,1,40.0
5,r0006_c0007,6,0.448272,0.448272,0.701630,0.958333,0.051001,0.003327,-7.570983,31.508390,...,0.000000,0.000000,0.162125,0,0,0,0,0,0,30.0
6,r0011_c0001,7,0.444945,0.418248,0.852840,0.950000,0.047675,0.001413,-7.634197,31.549407,...,0.120466,0.255454,0.571963,0,1,0,0,0,1,40.0
7,r0009_c0005,8,0.443532,0.443532,0.748574,0.941667,0.046262,0.004995,-7.592054,31.535340,...,0.000000,0.000000,0.054565,0,0,0,0,0,0,30.0
8,r0009_c0004,9,0.438538,0.438538,0.764730,0.933333,0.041267,0.001630,-7.602590,31.535340,...,0.000000,0.000000,0.009741,0,0,0,0,0,0,30.0
9,r0007_c0008,10,0.436907,0.436907,0.724915,0.925000,0.039637,0.000296,-7.560447,31.517373,...,0.000000,0.000000,0.366647,0,0,0,0,0,0,10.0


In [12]:
# 12) v5 sensitivity/stability analysis and seasonal stability

# A) Sensitivity analysis across ALL eligible grid cells.
all_gdf["s2_confidence"] = (all_gdf["s2_count"] / 8).clip(0, 1)

scenarios = {
    "balanced": {
        "visibility_score": 0.35,
        "remote_sensing_contrast": 0.35,
        "terrain_score": 0.20,
        "s2_confidence": 0.10,
    },
    "visibility_heavy": {
        "visibility_score": 0.55,
        "remote_sensing_contrast": 0.25,
        "terrain_score": 0.10,
        "s2_confidence": 0.10,
    },
    "contrast_heavy": {
        "visibility_score": 0.25,
        "remote_sensing_contrast": 0.55,
        "terrain_score": 0.10,
        "s2_confidence": 0.10,
    },
    "terrain_heavy": {
        "visibility_score": 0.25,
        "remote_sensing_contrast": 0.25,
        "terrain_score": 0.40,
        "s2_confidence": 0.10,
    },
}

for scenario_name, weights in scenarios.items():
    score_col = f"{scenario_name}_score"
    rank_col = f"{scenario_name}_rank"
    all_gdf[score_col] = 0.0
    for component, weight in weights.items():
        all_gdf[score_col] += all_gdf[component] * weight
    all_gdf[rank_col] = all_gdf[score_col].rank(
        ascending=False,
        method="min"
    ).astype(int)

rank_cols = [f"{name}_rank" for name in scenarios.keys()]

all_gdf["top10_count"] = sum((all_gdf[col] <= 10).astype(int) for col in rank_cols)
all_gdf["top25_count"] = sum((all_gdf[col] <= 25).astype(int) for col in rank_cols)
all_gdf["avg_rank"] = all_gdf[rank_cols].mean(axis=1)

all_gdf["stability_score"] = (
    all_gdf["top10_count"] * 2
    + all_gdf["top25_count"]
    - all_gdf["avg_rank"] / 25
)

# B) Seasonal stability.
# Seasonal scoring is intentionally simpler than the full score:
# visibility + spectral contrast + terrain + seasonal S2 coverage.
def month_filter(months):
    return ee.Filter.Or(*[ee.Filter.calendarRange(int(m), int(m), "month") for m in months])

def build_seasonal_score_fc(label, months):
    season_col = s2_col.filter(month_filter(months))
    image_count = season_col.size().getInfo()
    print(f"{label} Sentinel-2 image count:", image_count)

    if image_count == 0:
        return None

    season_s2 = season_col.median().clip(AOI)
    season_count = season_col.select("B4").count().rename(f"{label}_s2_count")

    season_ndvi = season_s2.normalizedDifference(["B8", "B4"]).rename(f"{label}_ndvi")
    season_bsi = (
        season_s2.expression(
            "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
            {
                "SWIR": season_s2.select("B11"),
                "RED": season_s2.select("B4"),
                "NIR": season_s2.select("B8"),
                "BLUE": season_s2.select("B2"),
            },
        )
        .rename(f"{label}_bsi")
    )

    season_lowveg = clamp01(ee.Image(0.55).subtract(season_ndvi).divide(0.55)).rename(f"{label}_lowveg")
    season_baresoil = clamp01(season_bsi.subtract(-0.10).divide(0.40)).rename(f"{label}_baresoil")
    season_visibility = (
        season_lowveg.multiply(0.55)
        .add(season_baresoil.multiply(0.45))
        .rename(f"{label}_visibility")
    )
    season_spectral_contrast = clamp01(
        local_abs_z(season_bsi, f"{label}_spectral_contrast", 90).divide(3.0)
    ).rename(f"{label}_spectral_contrast")
    season_s2_confidence = clamp01(season_count.divide(8)).rename(f"{label}_s2_confidence")

    season_score = (
        season_visibility.multiply(0.45)
        .add(season_spectral_contrast.multiply(0.35))
        .add(terrain_score.multiply(0.10))
        .add(season_s2_confidence.multiply(0.10))
        .updateMask(eligible_mask)
        .rename(f"{label}_season_score")
    )

    season_stack = ee.Image.cat([
        season_score,
        season_count,
        season_visibility,
        season_spectral_contrast,
        season_ndvi,
        season_bsi,
        eligible_mask,
    ]).clip(AOI)

    season_fc = (
        season_stack.reduceRegions(
            collection=grid,
            reducer=ee.Reducer.mean(),
            scale=30,
            tileScale=4,
        )
        .filter(ee.Filter.notNull([f"{label}_season_score"]))
        .filter(ee.Filter.gt("eligible_mask", 0))
        .sort(f"{label}_season_score", False)
    )

    return geemap.ee_to_gdf(season_fc).drop(columns=["geometry"], errors="ignore")

season_score_cols = []
season_rank_cols = []

for label, months in SEASON_MONTH_WINDOWS.items():
    season_table = build_seasonal_score_fc(label, months)
    if season_table is None or len(season_table) == 0:
        print(f"Skipped {label}: no valid seasonal rows")
        continue

    score_col = f"{label}_season_score"
    rank_col = f"{label}_season_rank"

    keep_cols = [
        "cell_id",
        score_col,
        f"{label}_s2_count",
        f"{label}_visibility",
        f"{label}_spectral_contrast",
        f"{label}_ndvi",
        f"{label}_bsi",
    ]
    season_table = season_table[keep_cols].copy()
    season_table[rank_col] = season_table[score_col].rank(
        ascending=False,
        method="min",
    ).astype(int)

    all_gdf = all_gdf.merge(season_table, on="cell_id", how="left")
    season_score_cols.append(score_col)
    season_rank_cols.append(rank_col)

if season_rank_cols:
    all_gdf["season_top10_count"] = sum((all_gdf[col] <= 10).fillna(False).astype(int) for col in season_rank_cols)
    all_gdf["season_top25_count"] = sum((all_gdf[col] <= 25).fillna(False).astype(int) for col in season_rank_cols)
    all_gdf["season_avg_rank"] = all_gdf[season_rank_cols].mean(axis=1)
    all_gdf["season_score_mean"] = all_gdf[season_score_cols].mean(axis=1)
    all_gdf["season_score_std"] = all_gdf[season_score_cols].std(axis=1).fillna(0)
else:
    all_gdf["season_top10_count"] = 0
    all_gdf["season_top25_count"] = 0
    all_gdf["season_avg_rank"] = None
    all_gdf["season_score_mean"] = None
    all_gdf["season_score_std"] = None

def norm01(series):
    series = pd.to_numeric(series, errors="coerce")
    min_v = series.min()
    max_v = series.max()
    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(0.5, index=series.index)
    return (series - min_v) / (max_v - min_v)

all_gdf["stability_score_norm"] = norm01(all_gdf["stability_score"])
all_gdf["season_stability_norm"] = norm01(all_gdf["season_top10_count"] * 2 + all_gdf["season_top25_count"])

# Final v5 review priority score.
# This is still a desk-review score, not evidence of treasure.
all_gdf["review_priority_score"] = (
    0.40 * all_gdf["candidate_score"]
    + 0.20 * all_gdf["confidence_score_all"]
    + 0.20 * all_gdf["stability_score_norm"]
    + 0.15 * all_gdf["season_stability_norm"]
    + 0.05 * norm01(all_gdf["score_gap_from_median"])
    - 0.08 * all_gdf["false_positive_warning_count"].clip(0, 3)
).clip(0, 1)

stable_cols = [
    "cell_id",
    "center_lon",
    "center_lat",
    "candidate_score",
    "quality_adjusted_score",
    "review_priority_score",
    "confidence_score_all",
    "stability_score",
    "top10_count",
    "top25_count",
    "avg_rank",
    "season_top10_count",
    "season_top25_count",
    "season_avg_rank",
    "season_score_mean",
    "season_score_std",
    "score_gap_from_median",
    "score_gap_to_next_rank",
    "balanced_rank",
    "visibility_heavy_rank",
    "contrast_heavy_rank",
    "terrain_heavy_rank",
    "visibility_score",
    "remote_sensing_contrast",
    "terrain_score",
    "s2_count",
    "builtup_frac",
    "builtup_near_frac",
    "cropland_frac",
    "water_edge_frac",
    "builtup_warning",
    "cropland_heavy_warning",
    "water_edge_warning",
    "modern_linear_edge_warning",
    "false_positive_warning_count",
    "worldcover_mode",
]

# Add seasonal rank/score columns to export if they exist.
for col in season_score_cols + season_rank_cols:
    if col not in stable_cols:
        stable_cols.append(col)

stable_candidates = all_gdf[stable_cols].sort_values(
    [
        "top10_count",
        "season_top10_count",
        "top25_count",
        "season_top25_count",
        "review_priority_score",
    ],
    ascending=False,
).reset_index(drop=True)

stable_candidates.head(25).to_csv("stable_candidate_priority_list_v5.csv", index=False)

# Enhanced top 25: same final prioritization, with geometry preserved.
top25_cell_ids = stable_candidates.head(TOP_N)["cell_id"].tolist()
top25_enhanced = all_gdf[all_gdf["cell_id"].isin(top25_cell_ids)].copy()
top25_enhanced["final_priority_rank"] = top25_enhanced["cell_id"].map(
    {cell_id: idx + 1 for idx, cell_id in enumerate(top25_cell_ids)}
)
top25_enhanced = top25_enhanced.sort_values("final_priority_rank").reset_index(drop=True)

top25_enhanced.drop(columns=["geometry"], errors="ignore").to_csv("top25_enhanced_v5.csv", index=False)
try:
    top25_enhanced.to_file("top25_enhanced_v5.geojson", driver="GeoJSON")
    print("Saved: top25_enhanced_v5.geojson")
except Exception as e:
    print("GeoJSON export warning:", e)

# Refresh all-cell quality diagnostics after stability/season columns were added.
all_gdf.drop(columns=["geometry"], errors="ignore").to_csv("quality_diagnostics_all_cells.csv", index=False)

print("Saved: stable_candidate_priority_list_v5.csv")
print("Saved: top25_enhanced_v5.csv")
print("Saved: quality_diagnostics_all_cells.csv")
stable_candidates.head(15)


dry_window Sentinel-2 image count: 6
cool_wet_window Sentinel-2 image count: 32
Saved: top25_enhanced_v5.geojson
Saved: stable_candidate_priority_list_v5.csv
Saved: top25_enhanced_v5.csv
Saved: quality_diagnostics_all_cells.csv


,cell_id,center_lon,center_lat,candidate_score,quality_adjusted_score,review_priority_score,confidence_score_all,stability_score,top10_count,top25_count,...,builtup_warning,cropland_heavy_warning,water_edge_warning,modern_linear_edge_warning,false_positive_warning_count,worldcover_mode,dry_window_season_score,cool_wet_window_season_score,dry_window_season_rank,cool_wet_window_season_rank
0,r0011_c0008,-7.560447,31.549407,0.474190,0.474190,0.748935,0.883793,11.86,4,4,...,0,0,0,0,0,30.0,0.540878,0.458616,1,2
1,r0008_c0008,-7.560447,31.526356,0.577309,0.542670,0.671133,0.851046,11.96,4,4,...,1,0,0,0,1,10.0,0.476819,0.459331,23,1
2,r0008_c0001,-7.634197,31.526356,0.454166,0.454166,0.680604,0.798140,8.51,3,3,...,0,0,0,0,0,30.0,0.484978,0.429298,7,10
3,r0009_c0004,-7.602590,31.535340,0.438538,0.438538,0.660660,0.764730,8.13,3,3,...,0,0,0,0,0,30.0,0.484842,0.444844,8,4
4,r0011_c0001,-7.634197,31.549407,0.444945,0.418248,0.570615,0.852840,9.70,3,4,...,1,0,0,0,1,40.0,0.533475,0.424745,2,13
5,r0008_c0007,-7.570983,31.526356,0.480380,0.480380,0.634131,0.795338,9.72,3,4,...,0,0,0,0,0,30.0,0.468708,0.442946,33,5
6,r0002_c0009,-7.552590,31.472458,0.449546,0.422573,0.525063,0.741423,9.63,3,4,...,0,1,0,0,1,40.0,0.475508,0.455766,26,3
7,r0009_c0005,-7.592054,31.535340,0.443532,0.443532,0.610933,0.748574,8.19,3,3,...,0,0,0,0,0,30.0,0.483105,0.441573,12,7
8,r0006_c0007,-7.570983,31.508390,0.448272,0.448272,0.555862,0.701630,8.33,3,3,...,0,0,0,0,0,30.0,0.478637,0.422594,20,14
9,r0006_c0000,-7.644732,31.508390,0.420298,0.420298,0.523246,0.712043,4.64,2,2,...,0,0,0,0,0,30.0,0.460793,0.442802,46,6


In [13]:
# 13) v6 stronger road/building filters and v6 priority score

# v6 adds stronger false-positive warnings without deleting candidates.
# Goal: reduce boring modern features before spending money on paid imagery.

import numpy as np

# Dynamic World is used as a second built-area signal, in addition to ESA WorldCover class 50.
# It is public GEE data and gives a built probability band. If no images are present, fall back to zeros.
dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
)

dw_count = dw_col.size().getInfo()
print("Dynamic World image count:", dw_count)

if dw_count > 0:
    dw_built_prob = dw_col.select("built").median().unmask(0).clip(AOI).rename("v6_dw_built_prob")
else:
    dw_built_prob = ee.Image(0).clip(AOI).rename("v6_dw_built_prob")

v6_dw_built_mask = dw_built_prob.gte(V6_DYNAMIC_WORLD_BUILT_PROB_THRESHOLD).rename("v6_dw_built_mask")
v6_strong_built_mask = builtup_mask.Or(v6_dw_built_mask).rename("v6_strong_built_mask")
v6_building_near_mask = (
    v6_strong_built_mask
    .focal_max(radius=200, units="meters")
    .rename("v6_building_near_mask")
)

# Road-like / modern corridor heuristic.
# This is not an official road layer. It flags strong linear/edge structure that may be roads, tracks,
# field boundaries, settlement edges, or other modern artifacts.
v6_optical_edge = ee.Image(
    ee.Algorithms.CannyEdgeDetector(
        image=bsi.unmask(0),
        threshold=0.35,
        sigma=1,
    )
).gt(0).rename("v6_optical_edge")

v6_sar_edge = ee.Image(
    ee.Algorithms.CannyEdgeDetector(
        image=vv_minus_vh.unmask(0),
        threshold=1.0,
        sigma=1,
    )
).gt(0).rename("v6_sar_edge")

v6_road_like_edge_mask = (
    v6_optical_edge
    .Or(v6_sar_edge)
    .And(low_vegetation.gt(0.35))
    .rename("v6_road_like_edge_mask")
)

v6_modern_corridor_mask = (
    v6_road_like_edge_mask
    .And(
        v6_building_near_mask
        .Or(cropland_mask)
        .Or(gentle_slope_score.gt(0.60))
    )
    .rename("v6_modern_corridor_mask")
)

v6_filter_stack = ee.Image.cat([
    dw_built_prob,
    v6_dw_built_mask.toFloat().rename("v6_dw_built_frac"),
    v6_strong_built_mask.toFloat().rename("v6_strong_built_frac"),
    v6_building_near_mask.toFloat().rename("v6_building_near_frac"),
    v6_road_like_edge_mask.toFloat().rename("v6_road_like_edge_frac"),
    v6_modern_corridor_mask.toFloat().rename("v6_modern_corridor_frac"),
]).clip(AOI)

v6_filter_fc = v6_filter_stack.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mean(),
    scale=10,
    tileScale=4,
)

v6_filter_gdf = geemap.ee_to_gdf(v6_filter_fc)
v6_filter_cols = [
    "cell_id",
    "v6_dw_built_prob",
    "v6_dw_built_frac",
    "v6_strong_built_frac",
    "v6_building_near_frac",
    "v6_road_like_edge_frac",
    "v6_modern_corridor_frac",
]

# Join v6 diagnostics onto all_gdf.
all_gdf = all_gdf.drop(columns=[c for c in v6_filter_cols if c != "cell_id"], errors="ignore")
all_gdf = all_gdf.merge(
    v6_filter_gdf[v6_filter_cols],
    on="cell_id",
    how="left",
)

for col in v6_filter_cols:
    if col != "cell_id":
        all_gdf[col] = pd.to_numeric(all_gdf[col], errors="coerce").fillna(0)

# v6 warning flags.
all_gdf["v6_building_warning"] = (
    (all_gdf["v6_strong_built_frac"] >= V6_STRONG_BUILT_FRAC)
    | (all_gdf["v6_building_near_frac"] >= V6_BUILDING_NEAR_FRAC)
).astype(int)

all_gdf["v6_road_like_warning"] = (
    (all_gdf["v6_road_like_edge_frac"] >= V6_ROAD_LIKE_EDGE_FRAC)
    | (all_gdf["v6_modern_corridor_frac"] >= V6_MODERN_CORRIDOR_FRAC)
).astype(int)

# Keep the v5 warnings and add the stronger v6 building/road-like warnings.
v6_warning_cols = [
    "v6_building_warning",
    "v6_road_like_warning",
    "cropland_heavy_warning",
    "water_edge_warning",
]

all_gdf["v6_false_positive_warning_count"] = all_gdf[v6_warning_cols].sum(axis=1)
all_gdf["v6_false_positive_penalty"] = (
    all_gdf["v6_false_positive_warning_count"] * 0.07
).clip(0, 0.28)

all_gdf["v6_quality_adjusted_score"] = (
    all_gdf["candidate_score"] * (1 - all_gdf["v6_false_positive_penalty"])
)

# v6 priority score: rank for paid imagery review, not proof of anything.
# It rewards good score, all-cell confidence, stability, seasonal stability, and low false-positive risk.
all_gdf["v6_no_warning_bonus"] = (1 - (all_gdf["v6_false_positive_warning_count"].clip(0, 4) / 4)).clip(0, 1)
all_gdf["v6_review_priority_score"] = (
    0.35 * norm01(all_gdf["v6_quality_adjusted_score"])
    + 0.25 * all_gdf["confidence_score_all"].clip(0, 1)
    + 0.20 * all_gdf["stability_score_norm"].clip(0, 1)
    + 0.10 * all_gdf["season_stability_norm"].clip(0, 1)
    + 0.05 * norm01(all_gdf["score_gap_from_median"])
    + 0.05 * all_gdf["v6_no_warning_bonus"]
).clip(0, 1)

v6_cols = [
    "cell_id",
    "center_lon",
    "center_lat",
    "candidate_score",
    "v6_quality_adjusted_score",
    "v6_review_priority_score",
    "confidence_score_all",
    "stability_score",
    "top10_count",
    "top25_count",
    "avg_rank",
    "season_top10_count",
    "season_top25_count",
    "season_avg_rank",
    "season_score_mean",
    "season_score_std",
    "score_gap_from_median",
    "score_gap_to_next_rank",
    "balanced_rank",
    "visibility_heavy_rank",
    "contrast_heavy_rank",
    "terrain_heavy_rank",
    "visibility_score",
    "remote_sensing_contrast",
    "terrain_score",
    "s2_count",
    "builtup_frac",
    "builtup_near_frac",
    "cropland_frac",
    "water_edge_frac",
    "v6_dw_built_prob",
    "v6_dw_built_frac",
    "v6_strong_built_frac",
    "v6_building_near_frac",
    "v6_road_like_edge_frac",
    "v6_modern_corridor_frac",
    "builtup_warning",
    "v6_building_warning",
    "v6_road_like_warning",
    "cropland_heavy_warning",
    "water_edge_warning",
    "modern_linear_edge_warning",
    "false_positive_warning_count",
    "v6_false_positive_warning_count",
    "worldcover_mode",
]

for col in season_score_cols + season_rank_cols:
    if col not in v6_cols and col in all_gdf.columns:
        v6_cols.append(col)

stable_candidates_v6 = all_gdf[v6_cols].sort_values(
    [
        "v6_false_positive_warning_count",
        "top10_count",
        "season_top10_count",
        "top25_count",
        "season_top25_count",
        "v6_review_priority_score",
    ],
    ascending=[True, False, False, False, False, False],
).reset_index(drop=True)

# Enhanced top 25 by v6 priority.
top25_v6_cell_ids = stable_candidates_v6.head(TOP_N)["cell_id"].tolist()
top25_enhanced_v6 = all_gdf[all_gdf["cell_id"].isin(top25_v6_cell_ids)].copy()
top25_enhanced_v6["final_priority_rank_v6"] = top25_enhanced_v6["cell_id"].map(
    {cell_id: idx + 1 for idx, cell_id in enumerate(top25_v6_cell_ids)}
)
top25_enhanced_v6 = top25_enhanced_v6.sort_values("final_priority_rank_v6").reset_index(drop=True)

stable_candidates_v6.head(25).to_csv("stable_candidate_priority_list_v6.csv", index=False)
top25_enhanced_v6.drop(columns=["geometry"], errors="ignore").to_csv("top25_enhanced_v6.csv", index=False)
all_gdf.drop(columns=["geometry"], errors="ignore").to_csv("quality_diagnostics_all_cells_v6.csv", index=False)

try:
    top25_enhanced_v6.to_file("top25_enhanced_v6.geojson", driver="GeoJSON")
    print("Saved: top25_enhanced_v6.geojson")
except Exception as e:
    print("GeoJSON export warning:", e)

print("Saved: stable_candidate_priority_list_v6.csv")
print("Saved: top25_enhanced_v6.csv")
print("Saved: quality_diagnostics_all_cells_v6.csv")
print("v6 strongest clean candidates:")
stable_candidates_v6.head(15)



Dynamic World image count: 54
Saved: top25_enhanced_v6.geojson
Saved: stable_candidate_priority_list_v6.csv
Saved: top25_enhanced_v6.csv
Saved: quality_diagnostics_all_cells_v6.csv
v6 strongest clean candidates:


,cell_id,center_lon,center_lat,candidate_score,v6_quality_adjusted_score,v6_review_priority_score,confidence_score_all,stability_score,top10_count,top25_count,...,cropland_heavy_warning,water_edge_warning,modern_linear_edge_warning,false_positive_warning_count,v6_false_positive_warning_count,worldcover_mode,dry_window_season_score,cool_wet_window_season_score,dry_window_season_rank,cool_wet_window_season_rank
0,r0000_c0009,-7.552590,31.454492,0.285709,0.285709,0.188829,0.295380,-4.62,0,0,...,0,0,0,0,0,20.0,0.271881,0.328369,117,106
1,r0000_c0008,-7.560447,31.454492,0.277536,0.277536,0.175955,0.298460,-4.71,0,0,...,0,0,0,0,0,20.0,0.262018,0.314649,120,112
2,r0009_c0004,-7.602590,31.535340,0.438538,0.407840,0.738672,0.764730,8.13,3,3,...,0,0,0,0,1,30.0,0.484842,0.444844,8,4
3,r0009_c0005,-7.592054,31.535340,0.443532,0.412485,0.709216,0.748574,8.19,3,3,...,0,0,0,0,1,30.0,0.483105,0.441573,12,7
4,r0006_c0000,-7.644732,31.508390,0.420298,0.390877,0.607516,0.712043,4.64,2,2,...,0,0,0,0,1,30.0,0.460793,0.442802,46,6
5,r0011_c0009,-7.552590,31.549407,0.410399,0.381671,0.596016,0.690378,1.14,1,1,...,0,0,0,0,1,30.0,0.511434,0.433267,4,9
6,r0003_c0003,-7.613125,31.481441,0.412531,0.383654,0.540569,0.650085,1.27,1,1,...,1,0,0,1,1,40.0,0.473742,0.434232,29,8
7,r0007_c0009,-7.552590,31.517373,0.419295,0.389944,0.501079,0.642619,1.49,1,1,...,0,0,0,0,1,10.0,0.389541,0.345521,99,93
8,r0003_c0005,-7.592054,31.481441,0.436611,0.406048,0.584300,0.719158,3.37,0,4,...,0,0,0,0,1,40.0,0.460204,0.419557,48,16
9,r0005_c0006,-7.581518,31.499407,0.425299,0.395528,0.536810,0.726550,2.00,0,3,...,0,0,0,0,1,30.0,0.463822,0.396928,39,37


In [14]:
# 14) v6 request-zone creation

# Request zones merge nearby high-priority cells into small quote-ready polygons.
# This avoids buying imagery for scattered single points.

import json
from math import cos, radians, sqrt


def approx_distance_m(lon1, lat1, lon2, lat2):
    mean_lat = radians((lat1 + lat2) / 2.0)
    dx = (lon2 - lon1) * 111_320.0 * cos(mean_lat)
    dy = (lat2 - lat1) * 111_320.0
    return sqrt(dx * dx + dy * dy)


def bbox_from_points(points, buffer_m):
    lons = [p[0] for p in points]
    lats = [p[1] for p in points]
    center_lat = sum(lats) / len(lats)
    dlat = buffer_m / 111_320.0
    dlon = buffer_m / (111_320.0 * max(0.2, cos(radians(center_lat))))
    return [
        min(lons) - dlon,
        min(lats) - dlat,
        max(lons) + dlon,
        max(lats) + dlat,
    ]


def polygon_from_bbox(bbox):
    west, south, east, north = bbox
    return {
        "type": "Polygon",
        "coordinates": [[
            [west, south],
            [east, south],
            [east, north],
            [west, north],
            [west, south],
        ]],
    }

# Prefer clean candidates, but fall back to the best v6 candidates if too few are clean.
zone_input = stable_candidates_v6.copy()
clean_zone_input = zone_input[zone_input["v6_false_positive_warning_count"] <= 1].copy()
if len(clean_zone_input) >= 3:
    zone_input = clean_zone_input

zone_input = zone_input.head(REQUEST_ZONE_TOP_CANDIDATES).reset_index(drop=True)

clusters = []
for _, row in zone_input.iterrows():
    point = (float(row["center_lon"]), float(row["center_lat"]))
    assigned = False
    for cluster in clusters:
        # Join cluster if point is close to any existing member.
        if any(
            approx_distance_m(point[0], point[1], member["lon"], member["lat"])
            <= REQUEST_ZONE_CLUSTER_DISTANCE_M
            for member in cluster["members"]
        ):
            cluster["members"].append({
                "cell_id": row["cell_id"],
                "lon": point[0],
                "lat": point[1],
                "v6_review_priority_score": float(row["v6_review_priority_score"]),
                "v6_false_positive_warning_count": int(row["v6_false_positive_warning_count"]),
            })
            assigned = True
            break
    if not assigned:
        clusters.append({
            "members": [{
                "cell_id": row["cell_id"],
                "lon": point[0],
                "lat": point[1],
                "v6_review_priority_score": float(row["v6_review_priority_score"]),
                "v6_false_positive_warning_count": int(row["v6_false_positive_warning_count"]),
            }]
        })

zone_records = []
features = []
for idx, cluster in enumerate(clusters, start=1):
    members = cluster["members"]
    points = [(m["lon"], m["lat"]) for m in members]
    bbox = bbox_from_points(points, REQUEST_ZONE_BUFFER_M)
    candidate_ids = [m["cell_id"] for m in members]
    best_member = max(members, key=lambda m: m["v6_review_priority_score"])
    zone_id = f"rz_v6_{idx:02d}"
    record = {
        "request_zone_id": zone_id,
        "primary_cell_id": best_member["cell_id"],
        "candidate_ids": ";".join(candidate_ids),
        "candidate_count": len(candidate_ids),
        "max_v6_review_priority_score": max(m["v6_review_priority_score"] for m in members),
        "mean_v6_review_priority_score": sum(m["v6_review_priority_score"] for m in members) / len(members),
        "max_v6_false_positive_warning_count": max(m["v6_false_positive_warning_count"] for m in members),
        "west": bbox[0],
        "south": bbox[1],
        "east": bbox[2],
        "north": bbox[3],
        "buffer_m": REQUEST_ZONE_BUFFER_M,
        "cluster_distance_m": REQUEST_ZONE_CLUSTER_DISTANCE_M,
    }
    zone_records.append(record)
    features.append({
        "type": "Feature",
        "properties": record,
        "geometry": polygon_from_bbox(bbox),
    })

request_zones_v6 = pd.DataFrame(zone_records).sort_values(
    ["max_v6_false_positive_warning_count", "max_v6_review_priority_score"],
    ascending=[True, False],
).head(REQUEST_ZONE_MAX_ZONES).reset_index(drop=True)

# Renumber after sorting/limiting and rebuild GeoJSON features from the final zone table.
for idx in range(len(request_zones_v6)):
    request_zones_v6.loc[idx, "request_zone_id"] = f"rz_v6_{idx + 1:02d}"

features = []
for _, zone in request_zones_v6.iterrows():
    bbox = [
        float(zone["west"]),
        float(zone["south"]),
        float(zone["east"]),
        float(zone["north"]),
    ]
    features.append({
        "type": "Feature",
        "properties": zone.drop(labels=[]).to_dict(),
        "geometry": polygon_from_bbox(bbox),
    })

request_zones_v6.to_csv("request_zones_v6.csv", index=False)
with open("request_zones_v6.geojson", "w", encoding="utf-8") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f, indent=2)

print("Saved: request_zones_v6.csv")
print("Saved: request_zones_v6.geojson")
print("Request zones:", len(request_zones_v6))
request_zones_v6




Saved: request_zones_v6.csv
Saved: request_zones_v6.geojson
Request zones: 6


,request_zone_id,primary_cell_id,candidate_ids,candidate_count,max_v6_review_priority_score,mean_v6_review_priority_score,max_v6_false_positive_warning_count,west,south,east,north,buffer_m,cluster_distance_m
0,rz_v6_01,r0000_c0009,r0000_c0009;r0000_c0008,2,0.188829,0.182392,0,-7.565186,31.450449,-7.547851,31.458534,450,1800
1,rz_v6_02,r0009_c0004,r0009_c0004;r0009_c0005,2,0.738672,0.723944,1,-7.607332,31.531297,-7.587311,31.539382,450,1800
2,rz_v6_03,r0006_c0000,r0006_c0000,1,0.607516,0.607516,1,-7.649474,31.504348,-7.639991,31.512433,450,1800
3,rz_v6_04,r0011_c0009,r0011_c0009,1,0.596016,0.596016,1,-7.557333,31.545365,-7.547846,31.553450,450,1800
4,rz_v6_05,r0003_c0005,r0003_c0005,1,0.584300,0.584300,1,-7.596794,31.477398,-7.587314,31.485483,450,1800
5,rz_v6_06,r0003_c0003,r0003_c0003;r0002_c0002,2,0.540569,0.540081,1,-7.628401,31.468415,-7.608385,31.485483,450,1800


In [15]:
# 15) v6 paid-imagery quote comparison template and scorer

# This cell does not order imagery.
# It creates a quote template and a scoring function for provider offers.

quote_template_rows = []
for _, zone in request_zones_v6.iterrows():
    quote_template_rows.append({
        "quote_id": "",
        "provider": "",
        "request_zone_id": zone["request_zone_id"],
        "primary_cell_id": zone["primary_cell_id"],
        "candidate_ids": zone["candidate_ids"],
        "archive_or_tasking": "archive",
        "acquisition_date": "",
        "resolution_m": "",
        "cloud_cover_pct": "",
        "off_nadir_deg": "",
        "price": "",
        "currency": "",
        "license_ok": "",          # TRUE/FALSE after license review
        "metadata_complete": "",   # TRUE/FALSE: sun angle, off-nadir, acquisition time, processing level
        "covers_requested_zone": "",# TRUE/FALSE after footprint check
        "notes": "",
    })

quote_template = pd.DataFrame(quote_template_rows)
quote_template.to_csv("paid_imagery_quote_template_v6.csv", index=False)


def _bool_score(value):
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if pd.isna(value):
        return 0.0
    return 1.0 if str(value).strip().lower() in {"true", "yes", "y", "1", "ok"} else 0.0


def compare_paid_imagery_quotes(input_csv="paid_imagery_quote_template_v6.csv"):
    quotes = pd.read_csv(input_csv)
    # Keep only rows where a provider or quote_id has actually been entered.
    entered = quotes[
        quotes["provider"].fillna("").astype(str).str.strip().ne("")
        | quotes["quote_id"].fillna("").astype(str).str.strip().ne("")
    ].copy()

    score_cols = [
        "resolution_score",
        "cloud_score",
        "off_nadir_score",
        "price_score",
        "license_score",
        "metadata_score",
        "coverage_score",
        "quote_score",
        "quote_decision",
    ]

    if entered.empty:
        empty = quotes.copy()
        for col in score_cols:
            empty[col] = ""
        empty.to_csv("paid_imagery_quote_comparison_v6.csv", index=False)
        print("Saved: paid_imagery_quote_template_v6.csv")
        print("Saved empty comparison: paid_imagery_quote_comparison_v6.csv")
        print("No provider quotes entered yet. Fill the template, then rerun compare_paid_imagery_quotes().")
        return empty

    for col in ["resolution_m", "cloud_cover_pct", "off_nadir_deg", "price"]:
        entered[col] = pd.to_numeric(entered[col], errors="coerce")

    entered["resolution_score"] = ((1.0 - entered["resolution_m"]) / 0.7).clip(0, 1).fillna(0)
    entered["cloud_score"] = (1 - (entered["cloud_cover_pct"] / 30.0)).clip(0, 1).fillna(0)
    entered["off_nadir_score"] = (1 - (entered["off_nadir_deg"] / 30.0)).clip(0, 1).fillna(0)

    if entered["price"].notna().sum() >= 2 and entered["price"].max() != entered["price"].min():
        entered["price_score"] = (1 - ((entered["price"] - entered["price"].min()) / (entered["price"].max() - entered["price"].min()))).clip(0, 1)
    else:
        entered["price_score"] = entered["price"].notna().astype(float) * 0.5

    entered["license_score"] = entered["license_ok"].apply(_bool_score)
    entered["metadata_score"] = entered["metadata_complete"].apply(_bool_score)
    entered["coverage_score"] = entered["covers_requested_zone"].apply(_bool_score)

    entered["quote_score"] = (
        0.25 * entered["resolution_score"]
        + 0.20 * entered["cloud_score"]
        + 0.15 * entered["off_nadir_score"]
        + 0.15 * entered["coverage_score"]
        + 0.10 * entered["price_score"]
        + 0.10 * entered["license_score"]
        + 0.05 * entered["metadata_score"]
    ).clip(0, 1)

    hard_fail = (
        (entered["license_score"] == 0)
        | (entered["coverage_score"] == 0)
        | (entered["metadata_score"] == 0)
    )
    entered["quote_decision"] = "review"
    entered.loc[hard_fail, "quote_decision"] = "do_not_order_until_fixed"
    entered.loc[(~hard_fail) & (entered["quote_score"] >= 0.75), "quote_decision"] = "best_candidate_quote"
    entered.loc[(~hard_fail) & (entered["quote_score"].between(0.55, 0.75, inclusive="left")), "quote_decision"] = "acceptable_quote"

    entered = entered.sort_values(["quote_decision", "quote_score"], ascending=[True, False]).reset_index(drop=True)
    entered.to_csv("paid_imagery_quote_comparison_v6.csv", index=False)
    print("Saved: paid_imagery_quote_comparison_v6.csv")
    return entered

quote_comparison_v6 = compare_paid_imagery_quotes()
quote_comparison_v6.head(20)



Saved: paid_imagery_quote_template_v6.csv
Saved empty comparison: paid_imagery_quote_comparison_v6.csv
No provider quotes entered yet. Fill the template, then rerun compare_paid_imagery_quotes().


,quote_id,provider,request_zone_id,primary_cell_id,candidate_ids,archive_or_tasking,acquisition_date,resolution_m,cloud_cover_pct,off_nadir_deg,...,notes,resolution_score,cloud_score,off_nadir_score,price_score,license_score,metadata_score,coverage_score,quote_score,quote_decision
0,NaN,NaN,rz_v6_01,r0000_c0009,r0000_c0009;r0000_c0008,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,
1,NaN,NaN,rz_v6_02,r0009_c0004,r0009_c0004;r0009_c0005,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,
2,NaN,NaN,rz_v6_03,r0006_c0000,r0006_c0000,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,
3,NaN,NaN,rz_v6_04,r0011_c0009,r0011_c0009,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,
4,NaN,NaN,rz_v6_05,r0003_c0005,r0003_c0005,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,
5,NaN,NaN,rz_v6_06,r0003_c0003,r0003_c0003;r0002_c0002,archive,NaN,NaN,NaN,NaN,...,NaN,,,,,,,,,


In [16]:
# 16) Create v6 paid archive request summary text

summary_name = "paid_archive_request_summary.txt"
top10_v6 = stable_candidates_v6.head(10)

with open(summary_name, "w", encoding="utf-8") as f:
    f.write("Paid Archive Imagery Request Summary — v6 Request Zones + Quote Comparison\n")
    f.write("=========================================================================\n\n")
    f.write(f"Generated UTC: {datetime.now(timezone.utc).isoformat()}\n\n")
    f.write("AOI bbox:\n")
    f.write(str(AOI_BBOX) + "\n\n")
    f.write("Date range used for free public triage:\n")
    f.write(f"{START_DATE} to {END_DATE}\n\n")
    f.write("Purpose:\n")
    f.write(
        "Lawful desk-based remote sensing triage only. "
        "This package ranks request zones and grid cells for possible paid archive imagery review. "
        "It does not authorize entry, excavation, collection, metal detecting, or field activity.\n\n"
    )
    f.write("Validation:\n")
    f.write(f"- Grid cells: {grid_count}\n")
    f.write(f"- Eligible ranked cells: {eligible_count}\n")
    f.write(f"- Exported candidate rows: {len(gdf)}\n")
    f.write(f"- All-cell v6 diagnostics rows: {len(all_gdf)}\n")
    f.write(f"- Request zones: {len(request_zones_v6)}\n")
    f.write(f"- Protected polygons intersecting AOI: {protected_count}\n")
    f.write(f"- Dynamic World image count for v6 built warning: {dw_count}\n\n")
    f.write("v6 upgrades included in this package:\n")
    f.write("- Request-zone creation from nearby stable priority cells.\n")
    f.write("- Stronger building warnings using ESA WorldCover built-up plus Dynamic World built probability.\n")
    f.write("- Stronger road/track-like warning using optical/SAR edge heuristics and modern-corridor context.\n")
    f.write("- Paid-imagery quote template and quote comparison scorer.\n\n")
    f.write("Top request zones:\n")
    f.write(request_zones_v6.to_csv(index=False))
    f.write("\nTop v6 candidate cells:\n")
    f.write(top10_v6.to_csv(index=False))
    f.write("\nPaid archive imagery specs to request:\n")
    f.write("- Optical resolution: 0.3 m to 1.0 m if budget allows.\n")
    f.write("- Cloud cover: less than 10%.\n")
    f.write("- Off-nadir angle: preferably less than 15 degrees.\n")
    f.write("- Coverage: request the request_zones_v6.geojson polygons, not only candidate center points.\n")
    f.write("- Season: match low-vegetation or bare-soil conditions from the seasonal diagnostics.\n")
    f.write("- Include metadata: acquisition date/time, sun angle, off-nadir angle, processing level, licensing terms.\n")
    f.write("- Compare offers with paid_imagery_quote_template_v6.csv and paid_imagery_quote_comparison_v6.csv.\n\n")
    f.write("Hard stop before field action:\n")
    f.write("- Written permission from landowner/manager.\n")
    f.write("- Required heritage/archaeology permits.\n")
    f.write("- Environmental and protected-area clearance.\n")
    f.write("- No excavation, collection, metal detecting, or disturbance without explicit authorization.\n\n")
    f.write("Interpretation warning:\n")
    f.write("- A high v6 score means better for paid imagery review, not treasure evidence.\n")
    f.write("- Road/building warnings are heuristic false-positive warnings and require visual review.\n")
    f.write("- Request zones are procurement zones for imagery, not field targets.\n")

print("Saved:", summary_name)



Saved: paid_archive_request_summary.txt


In [17]:
# 17) Export v6 map HTML and build final v6 ZIP package

# Add request zones to the map if possible.
try:
    import geopandas as gpd
    request_zones_gdf = gpd.read_file("request_zones_v6.geojson")
    Map.add_gdf(request_zones_gdf, layer_name="v6 request zones")
    print("Added v6 request zones to map")
except Exception as e:
    print("Request-zone map overlay warning:", e)

map_html = "visual_inspection_map.html"

try:
    Map.to_html(map_html)
    print("Saved map:", map_html)
except Exception as e:
    print("Map HTML export failed:", e)
    print("Keep a screenshot of Cell 10 manually if HTML export fails.")

files = []
files += glob.glob("lawful_gee_candidate_scout_top_*.csv")
files += glob.glob("lawful_gee_candidate_scout_top_*.geojson")
files += glob.glob("top25_enhanced_v5.csv")
files += glob.glob("top25_enhanced_v5.geojson")
files += glob.glob("quality_diagnostics_all_cells.csv")
files += glob.glob("stable_candidate_priority_list_v5.csv")
files += glob.glob("top25_enhanced_v6.csv")
files += glob.glob("top25_enhanced_v6.geojson")
files += glob.glob("quality_diagnostics_all_cells_v6.csv")
files += glob.glob("stable_candidate_priority_list_v6.csv")
files += glob.glob("request_zones_v6.csv")
files += glob.glob("request_zones_v6.geojson")
files += glob.glob("paid_imagery_quote_template_v6.csv")
files += glob.glob("paid_imagery_quote_comparison_v6.csv")
files += glob.glob("paid_archive_request_summary.txt")
files += glob.glob("visual_inspection_map.html")

# Remove duplicates while preserving order.
seen = set()
files = [f for f in files if not (f in seen or seen.add(f))]

zip_name = "paid_archive_request_candidate_package_FINAL_v6_ZONES_QUOTES.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for fpath in files:
        z.write(fpath)

print("Created:", zip_name)
print("Included files:")
for fpath in files:
    print("-", fpath)

print("\nFinal rule")
print("This notebook produces a desk-based paid imagery shortlist and request-zone package.")
print("It does not prove treasure, authorize entry, authorize metal detecting, authorize excavation, or replace permits.")
print("v6 adds request zones, stronger road/building false-positive warnings, and paid-imagery quote comparison.")



Added v6 request zones to map
Saved map: visual_inspection_map.html
Created: paid_archive_request_candidate_package_FINAL_v6_ZONES_QUOTES.zip
Included files:
- lawful_gee_candidate_scout_top_25_20260609T220648Z.csv
- lawful_gee_candidate_scout_top_25_20260609T220648Z.geojson
- top25_enhanced_v5.csv
- top25_enhanced_v5.geojson
- quality_diagnostics_all_cells.csv
- stable_candidate_priority_list_v5.csv
- top25_enhanced_v6.csv
- top25_enhanced_v6.geojson
- quality_diagnostics_all_cells_v6.csv
- stable_candidate_priority_list_v6.csv
- request_zones_v6.csv
- request_zones_v6.geojson
- paid_imagery_quote_template_v6.csv
- paid_imagery_quote_comparison_v6.csv
- paid_archive_request_summary.txt
- visual_inspection_map.html

Final rule
This notebook produces a desk-based paid imagery shortlist and request-zone package.
It does not prove treasure, authorize entry, authorize metal detecting, authorize excavation, or replace permits.
v6 adds request zones, stronger road/building false-positive war

## Final rule

This notebook produces a **desk-based paid imagery shortlist and request-zone package**.

It does **not** prove treasure, authorize entry, authorize metal detecting, authorize excavation, or replace permits.

v6 adds exactly these stage improvements:

1. Request-zone creation.
2. Stronger road/building false-positive warnings.
3. Paid-imagery quote comparison.

The output is for lawful desk review and imagery procurement only.
